*Módulo 2 de 9*

> **Prefer English?** Open [`02_how_a_satellite_sees.ipynb`](../en/02_how_a_satellite_sees.ipynb) — it is the same module, in English.


# 🛰️ Módulo 2 — Cómo ve el mundo un satélite

🧭 **Objetivos** — entender qué mide realmente un satélite, por qué una
imagen tiene muchas *bandas*, qué son la *reflectancia* y una *firma
espectral*, y las tres *resoluciones* que describen a cualquier sensor.
Luego abrir un tile **real** del Valle del Yaqui y confirmar que es
exactamente lo que prometió el Módulo 1: una cuadrícula de números.

📚 **La idea.** Un satélite lleva un **sensor** que mide cuánta luz solar
**refleja** el suelo de vuelta, en varias **bandas** — rebanadas angostas
del espectro electromagnético. Nuestros ojos ven tres bandas (rojo, verde,
azul). Satélites como **Landsat** y **Sentinel-2** ven esas *y* otras que no
podemos, en especial el **infrarrojo cercano (NIR)** y el **infrarrojo de
onda corta (SWIR)**, donde vegetación, suelo y agua más se diferencian.

El valor guardado para cada píxel y banda es la **reflectancia**: una
fracción entre 0 y 1 de la luz que rebotó (los archivos la guardan como
entero para ahorrar espacio, p. ej. `4500` = 0.45). Grafica la reflectancia
a lo largo de las bandas de un píxel y obtienes su **firma espectral** — una
huella que dice "esto es un cultivo próspero" o "esto es suelo desnudo".

![cómo ve un satélite](../../anim/es/01_where_it_runs.svg)


## Resolución: tres formas de decir "¿qué tan detallado?"

Todo sensor se describe con tres resoluciones — recuérdalas, deciden qué
puedes y qué no puedes mapear:

- **Espacial**: ¿qué tan grande es un píxel en el suelo? Nuestro tile es de
  **30 m** por píxel (un píxel ≈ una huerta pequeña). Más fina = parcelas
  más chicas visibles.
- **Espectral**: ¿cuántas bandas, y qué tan angostas? Más bandas = más
  química que puedes leer. Nuestro tile tiene **6 bandas espectrales** más
  índices derivados.
- **Temporal**: ¿cada cuánto revisita el satélite? Cada pocos días para
  Sentinel-2. Esto es lo que nos deja ver **crecer** un cultivo en la
  temporada.

📚 Los datos de abajo son una **geomediana** (el Módulo 3 la explica) de
marzo 2018, construida con **NASA HLS** — Landsat + Sentinel-2 armonizados —
a 30 m.


In [ ]:
# Trae el tile del taller (pocos MB; queda en caché tras la primera descarga)
import os, sys

async def trae_archivo(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await trae_archivo("crop_tile_384.tif")
print("Tile listo:", TILE)

## Abre el tile y lee su forma

Igual que el arreglo inventado de 8x8 del Módulo 1 — pero real. `rasterio`
abre archivos satelitales GeoTIFF; `.read()` nos da un arreglo NumPy con
forma `(bandas, filas, columnas)`.


In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(TILE) as src:
    img = src.read()                       # arreglo entero (13, 384, 384)
    nombres_banda = list(src.descriptions) # qué es cada capa

print("Forma del arreglo (bandas, filas, cols):", img.shape)
print("Tipo de dato:", img.dtype)
print("Las 13 capas:", nombres_banda)
print("Un píxel (fila 200, col 200), todas las bandas:", img[:, 200, 200])

## Míralo en color verdadero

Las primeras tres bandas espectrales son azul, verde, rojo. Apílalas (en
orden rojo, verde, azul para mostrar) y escala a 0–1, justo el truco de
dividir entre un número del Módulo 1. Este es el campo como lo verían tus
ojos desde el espacio.


In [ ]:
# Bandas 0,1,2 = azul, verde, rojo. Para mostrar se quiere Rojo-Verde-Azul.
rgb = np.clip(np.dstack([img[2], img[1], img[0]]) / 3000.0, 0, 1)

plt.figure(figsize=(7, 7))
plt.imshow(rgb)
plt.title("Valle del Yaqui — color verdadero (geomediana, marzo 2018)")
plt.axis("off")
plt.show()
print("Cada parcela que ves es un pedazo de píxeles de ~30 m.")

## La firma espectral: huellas de luz

Ahora el premio. Elige dos píxeles — uno sobre una parcela verde, otro sobre
suelo desnudo — y grafica su reflectancia en las 6 bandas espectrales. Las
curvas son sus **firmas espectrales**. Fíjate en el salto del cultivo hacia
la banda NIR: esa brecha es invisible a tus ojos pero obvia para el satélite,
y es toda la base de los índices de vegetación del Módulo 4.


In [ ]:
# Las 6 bandas espectrales son las capas 0..5 (azul,verde,rojo,nir,swir1,swir2)
espectrales = ["azul", "verde", "rojo", "nir", "swir1", "swir2"]

# El NDVI (capa 6) nos ayuda a hallar un píxel verde y uno desnudo automáticamente
ndvi = img[6] / 10000.0
verde_rc = np.unravel_index(np.argmax(ndvi), ndvi.shape)   # el más vegetado
suelo_rc = np.unravel_index(np.argmin(np.where(ndvi > 0, ndvi, 9)), ndvi.shape)

firma_cultivo = img[0:6, verde_rc[0], verde_rc[1]] / 10000.0
firma_suelo   = img[0:6, suelo_rc[0], suelo_rc[1]]  / 10000.0

plt.figure(figsize=(7, 4))
plt.plot(espectrales, firma_cultivo, marker="o", color="green", label="parcela verde")
plt.plot(espectrales, firma_suelo,  marker="s", color="peru",  label="suelo desnudo")
plt.ylabel("reflectancia"); plt.title("Firmas espectrales de dos píxeles reales")
plt.legend(); plt.show()
print("Ve cómo el cultivo salta en el NIR — eso es clorofila, no color.")

## 🧪 Ponte a prueba

**Tus ojos ven 3 bandas. ¿Por qué un satélite para mapear cultivos se
molesta en medir infrarrojo cercano y SWIR, que no podemos ver?**

<details><summary>Ver respuesta</summary>

Porque ahí es donde más difieren las superficies. La vegetación sana refleja
mucho NIR (por la estructura de la hoja) mientras absorbe el rojo; el suelo
y el agua se comportan distinto otra vez. Esas bandas invisibles cargan la
información que separa los cultivos de todo lo demás — los colores visibles
por sí solos no bastan.

</details>

**Un tile tiene 30 m de resolución espacial. ¿Qué significa ese número, y
qué resolución nos deja ver crecer un cultivo en la temporada?**

<details><summary>Ver respuesta</summary>

30 m de resolución espacial significa que cada píxel cubre un pedazo de
30 m × 30 m de suelo. Ver el crecimiento en el tiempo es la resolución
**temporal** — cada cuánto el satélite revisita el mismo lugar (cada pocos
días para Sentinel-2).

</details>


## 🔭 Profundiza

Opcional: estas tarjetas bilingües de conceptos amplían lo que acabas
de aprender (prerrequisitos, linaje a fundamentos, referencias):

- [Percepción remota, el campo mismo](https://abxda.github.io/rs-learning-audio/?id=remote-sensing&lang=es)
- [El espectro electromagnético](https://abxda.github.io/rs-learning-audio/?id=electromagnetic-spectrum&lang=es)
- [Reflectancia](https://abxda.github.io/rs-learning-audio/?id=reflectance&lang=es)
- [La firma espectral](https://abxda.github.io/rs-learning-audio/?id=spectral-signature&lang=es)
- [Bandas espectrales](https://abxda.github.io/rs-learning-audio/?id=spectral-bands&lang=es)
- [Resolución (espacial/espectral/temporal)](https://abxda.github.io/rs-learning-audio/?id=resolution&lang=es)
- [Landsat](https://abxda.github.io/rs-learning-audio/?id=landsat&lang=es)
- [Las misiones Sentinel](https://abxda.github.io/rs-learning-audio/?id=sentinel-missions&lang=es)



---

[← Anterior · Módulo 1 — El Python justo y necesario](01_python_justo.ipynb) · [Siguiente → · Módulo 3 — Datos limpios: de las nubes a la geomediana](03_datos_limpios_geomediana.ipynb)
